In [5]:
import pandas as pd 

In [6]:
data = pd.read_csv('mock_data.csv')
data.head()

,Mission_id,N_Recommandation,Recommandation,Evaluation,Recommendation_status,Action_id,Action,Échéance,Action_status
0,M0001,R0001,Mettre en place une politique de gestion des a...,Critique,accepté,A0001,Élaborer et diffuser la politique d'accès,2025-06-30,En cours
1,M0001,R0001,Mettre en place une politique de gestion des a...,Critique,accepté,A0002,Former les équipes IT sur la nouvelle politique,2025-07-15,Non échues
2,M0001,R0002,Renforcer les contrôles sur les journaux d'audit,Modéré,rejeté,NaN,NaN,NaN,NaN
3,M0001,R0003,Séparer les environnements de production et de...,Critique,accepté,A0003,Déployer un environnement de staging dédié,2025-08-01,En cours
4,M0001,R0004,Chiffrer les données sensibles en transit,Critique,accepté,A0004,Implémenter TLS 1.3 sur tous les services exposés,2025-09-01,Non échues


In [7]:
# global KPis 
data[["Recommendation_status","Recommandation"]].groupby(["Recommendation_status"]).nunique().reset_index()

,Recommendation_status,Recommandation
0,accepté,20
1,en etude,3
2,rejeté,3


##### **Recommandations**
- Nombre de recommandations non acceptées
- Nombre de recommandation à l’étude
- Répartition du nombre de recommandations acceptées  par criticité  (critique, modéré, mineur)
##### **Actions**
---
- Nombre total d’actions
- Répartition du nombre d’actions par criticité  (critique, modéré, mineur)
- Répartition par statut (non échues, en cours, clôturées, en continue, en retard, rééchelonné)

- Taux d’acceptation de recommandation = Nombre de recommandations acceptées / Nombre total de recommandations

- Taux de mise en œuvre des actions = Nombre d’actions réalisées / Nombre total d’actions

- Taux de mise en œuvre des actions critiques = Nombre d’actions critiques réalisées / Nombre total d’actions critiques




In [8]:
data.columns

Index(['Mission_id', 'N_Recommandation', 'Recommandation', 'Evaluation',
       'Recommendation_status', 'Action_id', 'Action', 'Échéance',
       'Action_status'],
      dtype='object')

In [9]:
data = pd.read_csv('mock_data.csv')

In [19]:
# Mission 
agg_dict = {
    f"Nb_{status}": (
        "N_Recommandation",
        lambda x, status=status: x[data.loc[x.index, "Recommendation_status"] == status ].nunique()
    )
    for status in data["Recommendation_status"].unique()
}

agg_dict["total_recommendation"] = (
    "N_Recommandation",
    "nunique"
)

tableau_1 = (
    data.groupby("Mission_id")
    .agg(**agg_dict)
    .reset_index()
)

tableau_1 

,Mission_id,Nb_accepté,Nb_rejeté,Nb_en etude,total_recommendation
0,M0001,3,1,0,4
1,M0002,3,0,1,4
2,M0003,4,1,0,5
3,M0004,4,0,1,5
4,M0005,6,1,1,8


In [ ]:
recommed_KPis = {}
recommed_KPis["Total_recom"] = tableau_1["total_recommendation"].sum().item()
recommed_KPis["Total_recom_ACC"] = tableau_1["Nb_accepté"].sum().item()
recommed_KPis["Total_recom_Rej"] = tableau_1["Nb_rejeté"].sum().item()
recommed_KPis["Total_recom_etu"] = tableau_1["Nb_en etude"].sum().item()
recommed_KPis["acceptation_rate"] = (round(recommed_KPis["Total_recom_ACC"]/recommed_KPis["Total_recom"],2)*100)
print(recommed_KPis["acceptation_rate"])

77.0


In [12]:
# recommendation accepte part evaluation
tableau_2 = (
    data.loc[data['Recommendation_status'] == "accepté",["N_Recommandation","Evaluation"]]
    .groupby("Evaluation")
    .agg(nb_recomendation=("N_Recommandation","nunique"))
    .reset_index()
)
tableau_2

,Evaluation,nb_recomendation
0,Critique,11
1,Mineur,2
2,Modéré,7


In [13]:
# Repartution action part criticité 
tableau_2 = (
    data[["Action_id","Evaluation"]]
    .groupby("Evaluation")
    .agg(nb_recomendation=("Action_id","nunique"))
    .reset_index()
)
tableau_2

,Evaluation,nb_recomendation
0,Critique,14
1,Mineur,2
2,Modéré,6


In [14]:
# Repartition action part Mission 
tableau_3 = (
    data[["Action_id","Mission_id"]]
    .groupby("Mission_id")
    .agg(nb_action=("Action_id","nunique"))
    .reset_index()
)
tableau_3

,Mission_id,nb_action
0,M0001,4
1,M0002,4
2,M0003,5
3,M0004,4
4,M0005,6


In [15]:
# Repartition  action part status 
tableau_4 = (
    data[['Action_id',"Action_status"]]
    .groupby("Action_status")
    .agg(nb_action=("Action_id","nunique"))
    .reset_index()
)
tableau_4



,Action_status,nb_action
0,Clôturées,3
1,En continue,2
2,En cours,4
3,En retard,3
4,Non échues,8
5,Rééchelonnées,1


In [16]:
results

,Mission_id,Nb_accepté,Nb_rejeté,Nb_en etude,total_recommendation
0,M0001,3,1,0,4
1,M0002,3,0,1,4
2,M0003,4,1,0,5
3,M0004,4,0,1,5
4,M0005,6,1,1,8


In [17]:
# Taux d’acceptation de recommandation :T1
T1 = round(results['Nb_accepté'].sum() / results['total_recommendation'].sum(),2) * 100
T1.item()

# Taux de mise en œuvre des actions : T2
d = tableau_4.set_index("Action_status")["nb_action"].to_dict()
T2 = round((d["En continue"] + d["Clôturées"]) / tableau_4.iloc[:,1].sum(), 2) * 100
T2.item()

# Taux de mise en œuvre des actions critiques : T3
# ACR : Action Crtitique Realisé
ACR = data[
    (data["Evaluation"] == "Critique") & 
    (
        (data["Action_status"] == "Clôturées") |
        (data["Action_status"] == "En cours")
    )
    ]['Action_id'].nunique()

T3 = round(ACR / tableau_4.iloc[:,1].sum(),3) * 100
T3.item()

28.599999999999998